In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import pandas as pd
import re
import joblib

from lightgbm import LGBMClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.calibration import CalibratedClassifierCV

In [2]:
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

In [ ]:
# df = pd.read_csv("/content/drive/MyDrive/parkinsons.data")

df = pd.read_csv("datasets\\speech\\oxford_PD_dataset\\parkinsons.data")
df.head()

,name,MDVP:Fo(Hz),MDVP:Fhi(Hz),MDVP:Flo(Hz),MDVP:Jitter(%),MDVP:Jitter(Abs),MDVP:RAP,MDVP:PPQ,Jitter:DDP,MDVP:Shimmer,...,Shimmer:DDA,NHR,HNR,status,RPDE,DFA,spread1,spread2,D2,PPE
0,phon_R01_S01_1,119.992,157.302,74.997,0.00784,0.00007,0.00370,0.00554,0.01109,0.04374,...,0.06545,0.02211,21.033,1,0.414783,0.815285,-4.813031,0.266482,2.301442,0.284654
1,phon_R01_S01_2,122.400,148.650,113.819,0.00968,0.00008,0.00465,0.00696,0.01394,0.06134,...,0.09403,0.01929,19.085,1,0.458359,0.819521,-4.075192,0.335590,2.486855,0.368674
2,phon_R01_S01_3,116.682,131.111,111.555,0.01050,0.00009,0.00544,0.00781,0.01633,0.05233,...,0.08270,0.01309,20.651,1,0.429895,0.825288,-4.443179,0.311173,2.342259,0.332634
3,phon_R01_S01_4,116.676,137.871,111.366,0.00997,0.00009,0.00502,0.00698,0.01505,0.05492,...,0.08771,0.01353,20.644,1,0.434969,0.819235,-4.117501,0.334147,2.405554,0.368975
4,phon_R01_S01_5,116.014,141.781,110.655,0.01284,0.00011,0.00655,0.00908,0.01966,0.06425,...,0.10470,0.01767,19.649,1,0.417356,0.823484,-3.747787,0.234513,2.332180,0.410335


In [4]:
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

In [5]:
groups = df["name"]

y = df["status"]
X = df.drop(columns=["name", "status"])

In [6]:
orig_cols = list(X.columns)

def clean_feature_names(columns):
    cleaned = []

    for col in columns:
        col = re.sub(r"[^A-Za-z0-9_]", "_", col)
        col = re.sub(r"_+", "_", col)
        col = col.strip("_")
        cleaned.append(col)

    return cleaned

X.columns = clean_feature_names(X.columns)

# Save mapping for SHAP later
feature_map = dict(zip(X.columns, orig_cols))

joblib.dump(
    feature_map,
    ARTIFACT_DIR / "feature_map.pkl"
)

['artifacts\\feature_map.pkl']

In [7]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

In [8]:
from sklearn.frozen import FrozenEstimator

base_model = LGBMClassifier(
    objective="binary",
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=15,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)
base_model.fit(X_train, y_train)

cal_model = CalibratedClassifierCV(
    estimator=FrozenEstimator(base_model),   # <-- wraps the fitted model, no refit
    method="sigmoid",
)
cal_model.fit(X_train, y_train)   # only fits the sigmoid calibration curve

probs = cal_model.predict_proba(X_test)[:, 1]
preds = (probs >= 0.5).astype(int)

[LightGBM] [Info] Number of positive: 115, number of negative: 41
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000701 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1118
[LightGBM] [Info] Number of data points in the train set: 156, number of used features: 22
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.737179 -> initscore=1.031360
[LightGBM] [Info] Start training from score 1.031360
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[

In [9]:
print("\n========== RESULTS ==========\n")

print(
    "ROC-AUC  :",
    round(roc_auc_score(y_test, probs), 4)
)

print(
    "Accuracy :",
    round(accuracy_score(y_test, preds), 4)
)

print(
    "Precision:",
    round(precision_score(y_test, preds), 4)
)

print(
    "Recall   :",
    round(recall_score(y_test, preds), 4)
)

print(
    "F1 Score :",
    round(f1_score(y_test, preds), 4)
)


========== RESULTS ==========

ROC-AUC  : 0.9286
Accuracy : 0.9487
Precision: 0.9412
Recall   : 1.0
F1 Score : 0.9697


In [19]:
joblib.dump(
    base_model,
    ARTIFACT_DIR / "speech_oxford_lgbm_v1.pkl"
)

joblib.dump(
    cal_model,
    ARTIFACT_DIR / "speech_oxford_lgbm_v1_calibrated.pkl"
)

print("\nSaved:")
print("speech_oxford_lgbm_v1.pkl")
print("speech_oxford_lgbm_v1_calibrated.pkl")
print("speech_oxford_feature_map.pkl")


Saved:
speech_oxford_lgbm_v1.pkl
speech_oxford_lgbm_v1_calibrated.pkl
speech_oxford_feature_map.pkl
